In [1]:
# =========================
#Imports & Spark
# =========================

import os
import sys
import requests
import pandas as pd
from calendar import monthrange

from pyspark.sql import SparkSession, functions as F

# Ensure Spark uses the current Python
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Point SPARK_HOME into the conda env pyspark install
conda_prefix = os.environ.get("CONDA_PREFIX")
spark_home = os.path.join(conda_prefix, "Lib", "site-packages", "pyspark")
os.environ["SPARK_HOME"] = spark_home

spark = (
    SparkSession.builder
    .appName("IND320_ELHUB_A4")
    .config(
        "spark.jars.packages",
        "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1"
    )
    .config("spark.cassandra.connection.host", "127.0.0.1")
    .config("spark.cassandra.connection.port", "9042")
    .config("spark.cassandra.connection.localDC", "datacenter1")
    .config("spark.driver.host", "127.0.0.1")
    .config(
        "spark.sql.extensions",
        "com.datastax.spark.connector.CassandraSparkExtensions"
    )
    .getOrCreate()
)

spark



In [2]:
# ======================================
#  Elhub helpers - PRODUCTION
# ======================================

def fetch_production_month(year: int, month: int) -> pd.DataFrame:
    """
    Fetch hourly PRODUCTION_PER_GROUP_MBA_HOUR for one month.
    """
    last_day = monthrange(year, month)[1]
    url = "https://api.elhub.no/energy-data/v0/price-areas"

    params = {
        "dataset": "PRODUCTION_PER_GROUP_MBA_HOUR",
        "startDate": f"{year}-{month:02d}-01T00:00:00+02:00",
        "endDate":   f"{year}-{month:02d}-{last_day}T23:00:00+02:00",
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    recs = []
    for pa in data.get("data", []):
        attrs = pa.get("attributes", {})
        recs.extend(attrs.get("productionPerGroupMbaHour", []))

    if not recs:
        return pd.DataFrame()

    df = pd.DataFrame(recs)
    df = df.rename(columns={
        "priceArea": "pricearea",
        "productionGroup": "productiongroup",
        "startTime": "starttime",
        "endTime": "endtime",
        "lastUpdatedTime": "lastupdatedtime",
        "quantityKwh": "quantitykwh",
    })

    return df


def write_production_year_to_cassandra(year: int):
    """
    Fetch 12 months for given year and write to Cassandra table elhub_production_2021.
    Re-running for the same year is safe: Cassandra upserts on primary key.
    """
    dfs = []

    for m in range(1, 13):
        pdf = fetch_production_month(year, m)
        print(f"PROD {year}-{m:02d}: {len(pdf)} rows")
        if not pdf.empty:
            dfs.append(pdf)

    if not dfs:
        print(f"No production data for {year}")
        return

    pdf_year = pd.concat(dfs, ignore_index=True)
    print(f"Total production rows for {year}: {len(pdf_year)}")

    sdf = spark.createDataFrame(pdf_year)

    sdf = (
        sdf.select(
            F.col("pricearea").cast("string").alias("pricearea"),
            F.col("productiongroup").cast("string").alias("productiongroup"),
            F.to_timestamp("starttime").alias("starttime"),
            F.to_timestamp("endtime").alias("endtime"),
            F.to_timestamp("lastupdatedtime").alias("lastupdatedtime"),
            F.col("quantitykwh").cast("double").alias("quantitykwh"),
        )
    )

    # Use append: Cassandra will upsert based on primary key
    (
        sdf.write
        .format("org.apache.spark.sql.cassandra")
        .options(table="elhub_production_2021", keyspace="ind320")
        .mode("append")
        .save()
    )

    print(f"✔ Production {year} written to Cassandra.")


In [ ]:
#==================================
#Run append production for 2021
#====================================
for year in [2022, 2023, 2024]:   
    write_production_year_to_cassandra(year)

In [3]:
#=================================================
#Sanity check after writing production in 2021-2024
#=====================================================
df_all = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(table="elhub_production_2021", keyspace="ind320")
    .load()
)

df_all.selectExpr("min(starttime) as min_ts", "max(starttime) as max_ts").show()

(
    df_all.selectExpr("year(starttime) as year")
    .groupBy("year")
    .count()
    .orderBy("year")
    .show()
)

+-------------------+-------------------+
|             min_ts|             max_ts|
+-------------------+-------------------+
|2020-12-31 23:00:00|2024-12-31 22:00:00|
+-------------------+-------------------+

+----+------+
|year| count|
+----+------+
|2020|    24|
|2021|215058|
|2022|218675|
|2023|218700|
|2024|219300|
+----+------+



In [4]:
# ======================================
# Elhub helpers - CONSUMPTION
# ======================================

def fetch_consumption_month(year: int, month: int) -> pd.DataFrame:
    """
    Fetch hourly CONSUMPTION_PER_GROUP_MBA_HOUR for one month.
    """
    last_day = monthrange(year, month)[1]
    url = "https://api.elhub.no/energy-data/v0/price-areas"

    params = {
        "dataset": "CONSUMPTION_PER_GROUP_MBA_HOUR",
        "startDate": f"{year}-{month:02d}-01T00:00:00+02:00",
        "endDate":   f"{year}-{month:02d}-{last_day}T23:00:00+02:00",
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    recs = []
    for pa in data.get("data", []):
        attrs = pa.get("attributes", {})
        recs.extend(attrs.get("consumptionPerGroupMbaHour", []))

    if not recs:
        return pd.DataFrame()

    df = pd.DataFrame(recs)
    df = df.rename(columns={
        "priceArea": "pricearea",
        "consumptionGroup": "consumptiongroup",
        "startTime": "starttime",
        "endTime": "endtime",
        "lastUpdatedTime": "lastupdatedtime",
        "quantityKwh": "quantitykwh",
    })

    return df


def write_consumption_year_to_cassandra(year: int):
    """
    Fetch 12 months for given year and write to Cassandra table elhub_consumption_2021_2024.
    """
    dfs = []

    for m in range(1, 13):
        pdf = fetch_consumption_month(year, m)
        print(f"CONS {year}-{m:02d}: {len(pdf)} rows")
        if not pdf.empty:
            dfs.append(pdf)

    if not dfs:
        print(f"No consumption data for {year}")
        return

    pdf_year = pd.concat(dfs, ignore_index=True)
    print(f"Total consumption rows for {year}: {len(pdf_year)}")

    sdf = spark.createDataFrame(pdf_year)

    sdf = (
        sdf.select(
            F.col("pricearea").cast("string").alias("pricearea"),
            F.col("consumptiongroup").cast("string").alias("consumptiongroup"),
            F.to_timestamp("starttime").alias("starttime"),
            F.to_timestamp("endtime").alias("endtime"),
            F.to_timestamp("lastupdatedtime").alias("lastupdatedtime"),
            F.col("quantitykwh").cast("double").alias("quantitykwh"),
        )
    )

    (
        sdf.write
        .format("org.apache.spark.sql.cassandra")
        .options(table="elhub_consumption_2021_2024", keyspace="ind320")
        .mode("append")
        .save()
    )

    print(f"✔ Consumption {year} written to Cassandra.")


In [ ]:
#========================================
#Run consumption append for 2021–2024
#========================================
for year in [2021, 2022, 2023, 2024]:
    write_consumption_year_to_cassandra(year)

2021-01: 18575 rows
2021-02: 16775 rows
2021-03: 18550 rows
2021-04: 17975 rows
2021-05: 18575 rows
2021-06: 17975 rows
2021-07: 18575 rows
2021-08: 18575 rows
2021-09: 17975 rows
2021-10: 18600 rows
2021-11: 17975 rows
2021-12: 18575 rows
Total rows for 2021: 218700
✔ Done: 2021 consumption appended to Cassandra.
2022-01: 18575 rows
2022-02: 16775 rows
2022-03: 18550 rows
2022-04: 17975 rows
2022-05: 18575 rows
2022-06: 17975 rows
2022-07: 18575 rows
2022-08: 18575 rows
2022-09: 17975 rows
2022-10: 18600 rows
2022-11: 17975 rows
2022-12: 18575 rows
Total rows for 2022: 218700
✔ Done: 2022 consumption appended to Cassandra.
2023-01: 18575 rows
2023-02: 16775 rows
2023-03: 18550 rows
2023-04: 17975 rows
2023-05: 18575 rows
2023-06: 17975 rows
2023-07: 18575 rows
2023-08: 18575 rows
2023-09: 17975 rows
2023-10: 18600 rows
2023-11: 17975 rows
2023-12: 18575 rows
Total rows for 2023: 218700
✔ Done: 2023 consumption appended to Cassandra.
2024-01: 18575 rows
2024-02: 17375 rows
2024-03: 185

In [5]:
from pyspark.sql import functions as F   # make sure F exists too

# Read back from Cassandra
df_cons = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(table="elhub_consumption_2021_2024", keyspace="ind320")
    .load()
)


In [6]:
#=====================
#sanity check consumption
#=======================
# min / max starttime
df_cons.selectExpr(
    "min(starttime) as min_ts",
    "max(starttime) as max_ts"
).show()

# rows per year
(
    df_cons
    .select(F.year("starttime").alias("year"))
    .groupBy("year")
    .count()
    .orderBy("year")
    .show()
)


+-------------------+-------------------+
|             min_ts|             max_ts|
+-------------------+-------------------+
|2021-01-01 00:00:00|2024-12-31 22:00:00|
+-------------------+-------------------+

+----+------+
|year| count|
+----+------+
|2021|218700|
|2022|218700|
|2023|218700|
|2024|219300|
+----+------+



In [7]:
# ======================================
#Read back from Cassandra
# ======================================

# Production (2021–2024)
df_prod = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(table="elhub_production_2021", keyspace="ind320")
    .load()
)

# Consumption (2021–2024)
df_cons = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(table="elhub_consumption_2021_2024", keyspace="ind320")
    .load()
)

df_prod.printSchema()
df_cons.printSchema()


root
 |-- pricearea: string (nullable = false)
 |-- productiongroup: string (nullable = false)
 |-- starttime: timestamp (nullable = true)
 |-- endtime: timestamp (nullable = true)
 |-- lastupdatedtime: timestamp (nullable = true)
 |-- quantitykwh: double (nullable = true)

root
 |-- pricearea: string (nullable = false)
 |-- consumptiongroup: string (nullable = false)
 |-- starttime: timestamp (nullable = true)
 |-- endtime: timestamp (nullable = true)
 |-- lastupdatedtime: timestamp (nullable = true)
 |-- quantitykwh: double (nullable = true)



In [ ]:
# MongoDB connection (single, final version)

from pymongo import MongoClient
from urllib.parse import quote_plus

USER = "anythingsane"
PWD  = "10987654321"
HOST = "cluster0.wvkiv4p.mongodb.net"

uri = (
    f"mongodb+srv://{quote_plus(USER)}:{quote_plus(PWD)}@{HOST}/"
    f"?retryWrites=true&w=majority&appName=Cluster0&authSource=admin"
)

client = MongoClient(uri)

db = client["elhub"]   # <--- use 'elhub' now for everything

prod_coll = db["production_mbahour"]
cons_coll = db["consumption_mbahour"]

print("✔ Connected to MongoDB")
print("Database:", db.name)
print("Collections:", db.list_collection_names())




In [ ]:
# ======================================================
# Export from Spark (df_prod / df_cons) to MongoDB
# ======================================================

import pandas as pd

def export_spark_df_to_mongo(df, coll, label, batch_size=10_000):
    """
    Export a Spark DataFrame to MongoDB using batch inserts.
    Assumes df is already loaded from Cassandra.
    """
    # 1) Spark -> pandas
    n_spark = df.count()
    print(f"[{label}] Spark rows:", n_spark)

    pdf = df.toPandas()
    n_pandas = len(pdf)
    print(f"[{label}] Pandas rows:", n_pandas)

    # 2) Pandas -> list of dicts
    records = pdf.to_dict("records")
    total = len(records)
    print(f"[{label}] Total records to insert: {total:,}")

    # 3) Batch insert
    inserted = 0
    for i in range(0, total, batch_size):
        batch = records[i : i + batch_size]
        coll.insert_many(batch)
        inserted += len(batch)
        print(f"[{label}] Inserted {inserted:,} / {total:,}")

    print(f"[{label}] ✔ Export finished. Final Mongo count:",
          coll.count_documents({}))


In [ ]:
# ======================
# RUN EXPORT
# ======================

print("Starting MongoDB export...\n")

export_spark_df_to_mongo(df_prod, prod_coll, "PRODUCTION")
export_spark_df_to_mongo(df_cons, cons_coll, "CONSUMPTION")

print("\n✔ All exports complete.")


In [ ]:
#==============================
#sanity test in mamghdb
#==============================
from pymongo import MongoClient
from urllib.parse import quote_plus

USER = "anythingsane"
PWD  = "10987654321"
HOST = "cluster0.wvkiv4p.mongodb.net"

uri = (
    f"mongodb+srv://{quote_plus(USER)}:{quote_plus(PWD)}@{HOST}/"
    f"?retryWrites=true&w=majority&appName=Cluster0&authSource=admin"
)

client = MongoClient(uri)
db = client["elhub"]

prod = db["production_mbahour"]
cons = db["consumption_mbahour"]

print("PROD docs:", prod.count_documents({}))
print("CONS docs:", cons.count_documents({}))

print("PROD date range:",
      prod.find_one(sort=[("starttime", 1)])["starttime"],
      "→",
      prod.find_one(sort=[("starttime", -1)])["starttime"])

print("CONS date range:",
      cons.find_one(sort=[("starttime", 1)])["starttime"],
      "→",
      cons.find_one(sort=[("starttime", -1)])["starttime"])


PROD docs: 871757
CONS docs: 875400
PROD date range: 2020-12-31 23:00:00 → 2024-12-31 22:00:00
CONS date range: 2021-01-01 00:00:00 → 2024-12-31 22:00:00


AI usage
I used AI tools (ChatGPT, cloud ai) to support parts of the development process. This included guidance when implementing STL decomposition, spectrogram analysis, SPC-based outlier detection, DCT transformations, and LOF anomaly detection. AI was also used to troubleshoot coding errors such as time-column mismatches, data-type issues, and general debugging challenges in Spark, Cassandra, MongoDB, and Streamlit. In addition, it helped with structuring tasks, clarifying statistical and signal-processing concepts, and improving the language of the project log.

Also used as a technical assistant while reorganising the Streamlit app and completing the analysis tasks. It helped me understand and organize  the page navigation, manage session tasks, and reuse functions across different parts of the project. I also used it to get idea about set up new streamlitt task, debugging and trouble shooting.


Project log
I configured a local Spark 3.5.1 environment using Java 17 and the correct Cassandra connector JAR, and verified that the PySpark driver points to the active conda environment. After confirming Cassandra 5 (Docker) was running, I created the tables for Elhub production and consumption datasets.

I implemented monthly data-fetch functions for both PRODUCTION_PER_GROUP_MBA_HOUR and CONSUMPTION_PER_GROUP_MBA_HOUR using the Elhub API, cleaned and standardized the fields, and wrote the data for 2021–2024 into Cassandra. I verified successful ingestion with sanity checks in Spark: minimum and maximum timestamps, yearly row counts, and schema validation.

After that, I set up a MongoDB Atlas connection and created a single database (elhub) with two collections: production_mbahour and consumption_mbahour. Using a custom batch-export function, I converted the Spark DataFrames to pandas and inserted the records into MongoDB in 10,000-document batches. Progress counters were used to confirm full export.

Finally, I performed a MongoDB sanity check by counting documents, confirming timestamp ranges, and ensuring that the data in MongoDB matched the data previously verified in Spark and Cassandra. All components (Spark → Cassandra → MongoDB) were successfully integrated and validated.

Streamlitt organised easy way with goble navigation:

🔧 IND320 Assignment 4
├── ⚙️ Global Settings
│   └── Data Years: [2021━━━━━━━━2024]
│       └── 📅 Active: 2021-2023
│
└── 📑 Navigation
    ├── SECTION 1: APP OVERVIEW
    │   └── ○ P1: Home
    │
    ├── SECTION 2: EXPLORATORY ANALYSIS
    │   ├── ○ P2: Weather Plots
    │   └── ○ P3: Energy Dashboard
    │
    ├── SECTION 3: STATISTICAL ANALYSIS
    │   ├── ○ P4: STL & Spectrogram
    │   ├── ○ P5: Outliers & Anomalies
    │   └── ○ P6: Weather–Energy Correlation
    │
    └── SECTION 4: PREDICTIVE MODELING
        ├── ○ P7: Regional Analysis
        └── ○ P8: Forecasting


Extream weather event 
High summer temperature decrease energy consumption.
High precipitation most time 

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt


# Load your data from MongoDB
# Assuming you have a DataFrame 'df' with columns:
# timestamp, wind_speed_10m, wind_gusts_10m, precipitation, 
# wind_production, hydro_production, thermal_production

# Define extreme weather thresholds
wind_speed_p95 = df['wind_speed_10m'].quantile(0.95)
wind_speed_p05 = df['wind_speed_10m'].quantile(0.05)
precip_p95 = df['precipitation'].quantile(0.95)
wind_gust_p95 = df['wind_gusts_10m'].quantile(0.95)

# Create extreme event masks
extreme_high_wind = df['wind_speed_10m'] > wind_speed_p95
extreme_low_wind = df['wind_speed_10m'] < wind_speed_p05
extreme_precipitation = df['precipitation'] > precip_p95
extreme_gusts = df['wind_gusts_10m'] > wind_gust_p95

# Normal conditions (between 25th and 75th percentile)
normal_conditions = (
    (df['wind_speed_10m'] >= df['wind_speed_10m'].quantile(0.25)) &
    (df['wind_speed_10m'] <= df['wind_speed_10m'].quantile(0.75)) &
    (df['precipitation'] <= df['precipitation'].quantile(0.75))
)

print("=" * 80)
print("CORRELATION ANALYSIS: NORMAL vs EXTREME CONDITIONS")
print("=" * 80)

# Function to calculate and display correlations
def analyze_correlations(mask, condition_name):
    subset = df[mask]
    print(f"\n{condition_name.upper()}")
    print(f"Number of observations: {len(subset)}")
    print("-" * 80)
    
    correlations = {
        'Wind Speed → Wind Production': 
            subset['wind_speed_10m'].corr(subset['wind_production']),
        'Wind Speed → Thermal Production': 
            subset['wind_speed_10m'].corr(subset['thermal_production']),
        'Wind Gusts → Wind Production': 
            subset['wind_gusts_10m'].corr(subset['wind_production']),
        'Wind Gusts → Thermal Production': 
            subset['wind_gusts_10m'].corr(subset['thermal_production']),
        'Precipitation → Hydro Production': 
            subset['precipitation'].corr(subset['hydro_production']),
        'Precipitation → Thermal Production': 
            subset['precipitation'].corr(subset['thermal_production']),
    }
    
    for relationship, corr_value in correlations.items():
        print(f"{relationship:45s}: {corr_value:7.4f}")
    
    return correlations

# Analyze different conditions
normal_corr = analyze_correlations(normal_conditions, "Normal Conditions")
high_wind_corr = analyze_correlations(extreme_high_wind, "Extreme High Wind Events")
low_wind_corr = analyze_correlations(extreme_low_wind, "Extreme Low Wind Events")
high_precip_corr = analyze_correlations(extreme_precipitation, "Extreme Precipitation Events")
gust_corr = analyze_correlations(extreme_gusts, "Extreme Wind Gust Events")

# Calculate correlation differences
print("\n" + "=" * 80)
print("CORRELATION CHANGES: EXTREME vs NORMAL CONDITIONS")
print("=" * 80)

def print_correlation_changes(extreme_corr, condition_name):
    print(f"\n{condition_name}:")
    print("-" * 80)
    for key in normal_corr.keys():
        diff = extreme_corr[key] - normal_corr[key]
        direction = "↑" if diff > 0 else "↓"
        print(f"{key:45s}: {direction} {abs(diff):6.4f} "
              f"({normal_corr[key]:.3f} → {extreme_corr[key]:.3f})")

print_correlation_changes(high_wind_corr, "Extreme High Wind")
print_correlation_changes(low_wind_corr, "Extreme Low Wind")
print_correlation_changes(high_precip_corr, "Extreme Precipitation")
print_correlation_changes(gust_corr, "Extreme Gusts")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Wind Speed vs Wind Production: Normal vs Extreme
ax1 = axes[0, 0]
ax1.scatter(df[normal_conditions]['wind_speed_10m'], 
           df[normal_conditions]['wind_production'], 
           alpha=0.3, s=20, label='Normal', color='blue')
ax1.scatter(df[extreme_high_wind]['wind_speed_10m'], 
           df[extreme_high_wind]['wind_production'], 
           alpha=0.5, s=30, label='Extreme High Wind', color='red')
ax1.set_xlabel('Wind Speed (m/s)')
ax1.set_ylabel('Wind Production (normalized)')
ax1.set_title('Wind Speed → Wind Production')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Wind Speed vs Thermal Production
ax2 = axes[0, 1]
ax2.scatter(df[normal_conditions]['wind_speed_10m'], 
           df[normal_conditions]['thermal_production'], 
           alpha=0.3, s=20, label='Normal', color='blue')
ax2.scatter(df[extreme_high_wind]['wind_speed_10m'], 
           df[extreme_high_wind]['thermal_production'], 
           alpha=0.5, s=30, label='Extreme High Wind', color='red')
ax2.set_xlabel('Wind Speed (m/s)')
ax2.set_ylabel('Thermal Production (normalized)')
ax2.set_title('Wind Speed → Thermal Production (Compensation)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Precipitation vs Hydro Production
ax3 = axes[1, 0]
ax3.scatter(df[normal_conditions]['precipitation'], 
           df[normal_conditions]['hydro_production'], 
           alpha=0.3, s=20, label='Normal', color='blue')
ax3.scatter(df[extreme_precipitation]['precipitation'], 
           df[extreme_precipitation]['hydro_production'], 
           alpha=0.5, s=30, label='Extreme Precipitation', color='green')
ax3.set_xlabel('Precipitation (mm)')
ax3.set_ylabel('Hydro Production (normalized)')
ax3.set_title('Precipitation → Hydro Production')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Time series showing thermal stability
ax4 = axes[1, 1]
# Calculate rolling correlation
window_size = 168  # 1 week window
rolling_corr_wind = df['wind_speed_10m'].rolling(window_size).corr(df['wind_production'])
rolling_corr_thermal = df['wind_speed_10m'].rolling(window_size).corr(df['thermal_production'])

ax4.plot(df['timestamp'], rolling_corr_wind, label='Wind Speed → Wind Prod', alpha=0.7)
ax4.plot(df['timestamp'], rolling_corr_thermal, label='Wind Speed → Thermal Prod', alpha=0.7)
ax4.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax4.set_xlabel('Time')
ax4.set_ylabel('Rolling Correlation (7-day window)')
ax4.set_title('Temporal Variation in Correlations')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('extreme_weather_correlation_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "=" * 80)
print("Visualization saved as 'extreme_weather_correlation_analysis.png'")
print("=" * 80)

ModuleNotFoundError: No module named 'seaborn'